In [1]:
# import getpass
import os

# os.environ["OPENAI_API_KEY"] = getpass.getpass()
os.environ["OPENAI_API_KEY"] = "anything"

In [4]:
# 单轮对话
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="llama3.2:3b",base_url="http://localhost:11434/v1")

from langchain_core.messages import HumanMessage


resp = model.invoke([HumanMessage(content="Hi! I'm Bob")])
print(resp)

content="Hi Bob! It's nice to meet you. Is there something I can help you with, or would you like to chat for a bit?" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 30, 'total_tokens': 60, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'finish_reason': 'stop', 'logprobs': None} id='run-7e12dbd1-4deb-4a6e-9f72-938288010e6f-0' usage_metadata={'input_tokens': 30, 'output_tokens': 30, 'total_tokens': 60, 'input_token_details': {}, 'output_token_details': {}}


In [6]:
# 多轮对话，引入历史对话
from langchain_core.messages import AIMessage
resp = model.invoke(
    [
        HumanMessage(content="Hi! I'm Bob"),
        AIMessage(content="Hello Bob! How can I assist you today?"),
        HumanMessage(content="What's my name?"),
    ]
)

print(resp)

content='Sorry, I don\'t actually know your real name. You said "hi, I\'m Bob" when we started chatting, but that was just a playful introduction to get the conversation started. I don\'t have any other information about you beyond that. Would you like to share your actual name with me?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 55, 'total_tokens': 117, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'finish_reason': 'stop', 'logprobs': None} id='run-c70cb947-2981-4b69-8afe-47b507c44df8-0' usage_metadata={'input_tokens': 55, 'output_tokens': 62, 'total_tokens': 117, 'input_token_details': {}, 'output_token_details': {}}


In [7]:
# 消息历史
# 我们可以使用消息历史类来包装我们的模型，使其具有状态。 这将跟踪模型的输入和输出，并将其存储在某个数据存储中。 未来的交互将加载这些消息，并将其作为输入的一部分传递给链
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model, get_session_history)

config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Bob")],
    config=config,
)

response.content

"Hello, Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [ ]:
# 使用同一configurable对象，我们可以继续进行对话
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'I remember! Your name is Bob! We just started our conversation, and I said it was nice to meet you when you introduced yourself by your name.'

In [ ]:
# 使用新configurable对象，我们可以继续进行对话
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

"I don't know your name. I'm a large language model, I don't have the ability to store or access personal information about individual users. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. Would you like to share your name with me?"

In [10]:
# 提示词模板
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

response = chain.invoke({"messages": [HumanMessage(content="hi! I'm bob")]})

response.content

'Hi Bob! Nice to meet you. Is there something I can help you with, or would you like to chat for a bit?'

In [11]:
# 现在可以将其包装在与之前相同的消息历史对象中
with_message_history = RunnableWithMessageHistory(chain, get_session_history)
config = {"configurable": {"session_id": "abc5"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Jim")],
    config=config,
)

response.content

"Hello, Jim! It's great to meet you! Is there something I can help you with or would you like to chat for a bit?"

In [12]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Jim. You told me that earlier.'

In [13]:
# 更复杂的提示模板
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

response = chain.invoke(
    {"messages": [HumanMessage(content="hi! I'm bob")], "language": "Spanish"}
)

response.content

'Hola Bob! (Hello, Bob!) ¿En qué puedo ayudarte today? (How can I help you today?)'

In [14]:
# 将这个更复杂的链封装在一个消息历史类中。这次，由于输入中有多个键，我们需要指定正确的键来保存聊天历史。
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config = {"configurable": {"session_id": "abc11"}}

response = with_message_history.invoke(
    {"messages": [HumanMessage(content="hi! I'm todd")], "language": "Spanish"},
    config=config,
)

response.content

"Hola Todd, ¿cómo estás? Estoy aquí para ayudarte con cualquier cosa que necesites. ¿En qué puedo ayudarte hoy?\n(Hello Todd, how are you? I'm here to help you with anything you need. How can I help you today?)"